# BP: follow a boundary through the orbit quotient

**Question.** How is a binary boundary induced on orbit classes, and why must each source orbit be processed exactly once?

**Paper locations.** Section VIII; Appendix G.4, “BP quotient maps and construction algorithm,” Algorithm 2; and Appendix G.5, “Examples of BP codes” (`sec:BP`, `app:bp-checks`, `app:bp-examples`).

**Companion paper.** Yuan Yao, Guo Zhang, Yingli Yang, Yuanye Zhu, Xiaosi Xu, Xiao Yuan, and Ying Li, **“An introduction to qLDPC codes from a structural perspective.”** Submitted.

The paper numbers refer to the manuscript identified by `PAPER_SOURCE` in
[`paper_cases.py`](../qldpc_generator/paper_cases.py). Source labels are retained
to help locate the same material after renumbering. Equation numbers below are
local to this notebook; code outputs are computational results.

**Before running.** Follow the [installation instructions](../README.md#installation-and-execution),
select that Python environment, and run the cells in order. All matrix arithmetic
and ranks are over GF(2); matrix indices start at zero. For the coverage overview,
start with [06 — Paper validation](06-paper-validation.ipynb).

In [1]:
import numpy as np
from IPython.display import Markdown, display

def show_table(headers, rows):
    """Format results for reading; no mathematical calculations happen here."""
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

## 1. Define the quotient before computing a matrix

Let C and D be two-term binary chain complexes, with a right action on C and
a left action on D. The actions must commute with the respective boundaries.
On basis pairs the balanced relation and induced boundary are

$$
(c\cdot g,d)\sim(c,g\cdot d),\qquad
(c,d)\sim(c\cdot g,g^{-1}\cdot d),\qquad
\partial[c,d]=[\partial_C c,d]+[c,\partial_D d]. \tag{1}
$$

The quotient has one basis vector per orbit, not a normalized sum of the
vectors in that orbit. Choose one representative, apply its boundary, then
collect target orbits modulo two. No division by the group order is involved.

For the smallest useful example, take G = C2. It swaps the two C basis vectors
in each degree, while both D spaces have one fixed basis vector and zero
boundary. We compare two equivariant C boundaries:

$$
d_C^{\rm one}=\begin{pmatrix}1&0\\0&1\end{pmatrix},\qquad
d_C^{\rm two}=\begin{pmatrix}1&1\\1&1\end{pmatrix},\qquad d_D=(0). \tag{2}
$$

An action array lists the image of every basis index for each group element.
The D action is nonfree, which is allowed for BP itself.

In [2]:
from qldpc_generator import balanced_product
from qldpc_generator.algebra import cyclic_group, polynomial_to_circulant, x

group = cyclic_group(2)
swap_action = group.regular()
fixed_action = np.zeros((2, 1), dtype=int)
one = balanced_product(np.eye(2, dtype=np.uint8), [[0]], group,
                       (swap_action, swap_action), (fixed_action, fixed_action))
two = balanced_product(np.ones((2, 2), dtype=np.uint8), [[0]], group,
                       (swap_action, swap_action), (fixed_action, fixed_action))
print("C2 action (identity row, then swap row):", swap_action.tolist())
print("D action:", fixed_action.tolist())
show_table(["Degree (C,D)", "Pairs in each orbit", "Chosen representatives"], [
    [degree, space.orbits, space.representatives] for degree, space in one.spaces.items()
])

C2 action (identity row, then swap row): [[0, 1], [1, 0]]
D action: [[0], [0]]


| Degree (C,D) | Pairs in each orbit | Chosen representatives |
| --- | --- | --- |
| (0, 0) | [[(0, 0), (1, 0)]] | [(0, 0)] |
| (0, 1) | [[(0, 0), (1, 0)]] | [(0, 0)] |
| (1, 0) | [[(0, 0), (1, 0)]] | [(0, 0)] |
| (1, 1) | [[(0, 0), (1, 0)]] | [(0, 0)] |

## 2. Trace one source column by hand

Consider the source orbit in degree (1,0). Its representative is (0,0).
With the identity boundary there is one target term. With the all-ones
boundary there are two terms, but both land in the same target orbit:

$$
\partial_C^{\rm one}[0,0]=[0,0],\qquad
\partial_C^{\rm two}[0,0]=[0,0]+[1,0]=0. \tag{3}
$$

The last equality holds in the quotient over GF(2), because (0,0) and (1,0)
represent the same orbit. The trace below displays the actual pairs before
combining them. The qubit order is degree (0,1), then degree (1,0).

In [3]:
trace_rows = []
for name, bp in [('one term', one), ('two terms', two)]:
    c, d = bp.spaces[1, 0].representatives[0]
    target_pairs = [(int(v), d) for v in np.flatnonzero(bp.dc[:, c])]
    target_orbits = [int(bp.spaces[0, 0].index[v, w]) for v, w in target_pairs]
    column = np.bincount(target_orbits, minlength=len(bp.spaces[0, 0].orbits)) % 2
    offset = len(bp.spaces[0, 1].orbits)
    assert np.array_equal(column, bp.code.hx[:, offset])
    trace_rows.append([name, (c, d), target_pairs, target_orbits, column.tolist()])
show_table(["Boundary", "Source representative", "Target pairs", "Target orbit indices", "Induced column"], trace_rows)
assert one.code.hx.tolist() == [[0, 1]]
assert two.code.hx.tolist() == [[0, 0]]
show_table(["Example", "H_X", "H_Z"], [
    ["one term", one.code.hx.tolist(), one.code.hz.tolist()],
    ["two cancelling terms", two.code.hx.tolist(), two.code.hz.tolist()],
])

| Boundary | Source representative | Target pairs | Target orbit indices | Induced column |
| --- | --- | --- | --- | --- |
| one term | (0, 0) | [(0, 0)] | [0] | [1] |
| two terms | (0, 0) | [(0, 0), (1, 0)] | [0, 0] | [0] |

| Example | H_X | H_Z |
| --- | --- | --- |
| one term | [[0, 1]] | [[1, 0]] |
| two cancelling terms | [[0, 0]] | [[0, 0]] |

### Two failure controls

Summing the induced column over every source representative counts the same
quotient basis vector twice and incorrectly erases the nonzero identity
boundary. Conversely, combining target terms with Boolean OR instead of XOR
incorrectly retains the two cancelling terms. Neither operation constructs
the balanced boundary.

In [4]:
wrong_source_sum = np.zeros(1, dtype=np.uint8)
for c, d in one.spaces[1, 0].orbits[0]:
    for v in np.flatnonzero(one.dc[:, c]):
        wrong_source_sum[one.spaces[0, 0].index[v, d]] ^= 1
wrong_target_or = np.zeros(1, dtype=np.uint8)
c, d = two.spaces[1, 0].representatives[0]
for v in np.flatnonzero(two.dc[:, c]):
    wrong_target_or[two.spaces[0, 0].index[v, d]] = 1
assert wrong_source_sum.tolist() == [0] and one.code.hx[:, 1].tolist() == [1]
assert wrong_target_or.tolist() == [1] and two.code.hx[:, 1].tolist() == [0]
show_table(["Rule", "Wrong column", "Correct column"], [
    ["Sum every source representative", wrong_source_sum.tolist(), [1]],
    ["OR repeated target incidences", wrong_target_or.tolist(), [0]],
])

| Rule | Wrong column | Correct column |
| --- | --- | --- |
| Sum every source representative | [0] | [1] |
| OR repeated target incidences | [1] | [0] |

## 3. Give all inputs for BP90 and BP108

The paper's larger examples use cyclic polynomials of period 3q. The acting
group is the subgroup generated by a shift of three, so it has order q:

$$
p_1=1+x+x^5,\qquad
p_2=\begin{cases}1+x^2+x^7,&q=5,\\1+x+x^2,&q=6,\end{cases}
\qquad g\cdot i=i+3g\pmod{3q}. \tag{4}
$$

These cyclic actions can be used on either side because the group is abelian.
In each chain degree the unquotiented pair basis has (3q)² elements, partitioned
into free orbits of size q. Thus each quotient degree has 9q basis vectors,
and the two qubit sectors together have N = 18q.

The constructor below takes the polynomials and actions explicitly. We also
change every representative from the first to the last pair in its orbit.
The orbit numbering stays fixed, so the resulting check matrices should be
identical entry by entry.

In [5]:
from qldpc_generator.paper_cases import cyclic_bp_case
instances = {}
rows = []
for q in (5, 6):
    period = 3*q
    p1 = 1+x+x**5
    p2 = 1+x**2+x**7 if q == 5 else 1+x+x**2
    action = np.array([[(i+3*g) % period for i in range(period)] for g in range(q)])
    dc = polynomial_to_circulant(p1, period)
    dd = polynomial_to_circulant(p2, period)
    args = (dc, dd, cyclic_group(q), (action, action), (action, action))
    bp = balanced_product(*args)
    last = balanced_product(*args, representative='last')
    stored = cyclic_bp_case(q)
    for other in (last, stored):
        assert np.array_equal(bp.code.hx, other.code.hx)
        assert np.array_equal(bp.code.hz, other.code.hz)
    assert all(len(space.orbits) == 9*q for space in bp.spaces.values())
    assert (bp.code.n, bp.code.k) == (18*q, 8)
    instances[q] = bp
    p = bp.code.parameters()
    rows.append([q, str(p1), str(p2), dc.shape, 9*q, bp.code.n, bp.code.k, p['rank_x']])
show_table(["q", "p1", "p2", "Factor shape", "Orbits per degree", "N", "K", "Each rank"], rows)
print("Sample q=5 orbit:", instances[5].spaces[0, 0].orbits[0])

| q | p1 | p2 | Factor shape | Orbits per degree | N | K | Each rank |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 5 | x**5 + x + 1 | x**7 + x**2 + 1 | (15, 15) | 45 | 90 | 8 | 41 |
| 6 | x**5 + x + 1 | x**2 + x + 1 | (18, 18) | 54 | 108 | 8 | 50 |

Sample q=5 orbit: [(0, 0), (3, 12), (6, 9), (9, 6), (12, 3)]


## 4. Cross-check the folded matrices printed in the paper

Appendix G.5 uses representatives whose first exponent is 0, 1, or 2. In that
ordering, with each polynomial entry expanded at period 3q, the blocks are

$$
A=\mathbb B_{3q}\!\begin{pmatrix}1&x^6&x^3\\1&1&x^6\\x^3&1&1\end{pmatrix},
\quad B=I_3\otimes\mathbb B_{3q}(p_2),\quad
H_X=(B\;A),\quad H_Z=(A^{\mathsf T}\;B^{\mathsf T}). \tag{5}
$$

This comparison uses the independently specified folded blocks, rather than
merely comparing two calls to the orbit constructor.

In [6]:
for q, bp in instances.items():
    period = 3*q
    pc = lambda polynomial: polynomial_to_circulant(polynomial, period)
    p2 = 1+x**2+x**7 if q == 5 else 1+x+x**2
    A = np.block([[pc(1), pc(x**6), pc(x**3)],
                  [pc(1), pc(1), pc(x**6)],
                  [pc(x**3), pc(1), pc(1)]])
    B = np.kron(np.eye(3, dtype=np.uint8), pc(p2))
    assert np.array_equal(bp.code.hx, np.hstack([B, A]))
    assert np.array_equal(bp.code.hz, np.hstack([A.T, B.T]))
    print(f"q={q}: both folded matrices agree; commutator nonzeros =",
          np.count_nonzero(bp.code.hx @ bp.code.hz.T % 2))

q=5: both folded matrices agree; commutator nonzeros = 0
q=6: both folded matrices agree; commutator nonzeros = 0


## 5. Report a lower bound as a lower bound

The paper quotes distances 10 and 8. The bounded exhaustive searches below
exclude every nontrivial logical support through weight six. They therefore
establish D >= 7 for each instance, and do not reproduce the quoted exact
distances. Changing representatives checks the construction, not its distance.

In [7]:
rows = []
for q, bp in instances.items():
    result = bp.code.distance(max_weight=6)
    assert result['status'] == 'bounded' and result['lower_bound'] == 7
    rows.append([f"BP{bp.code.n}", 10 if q == 5 else 8,
                 f"D >= {result['lower_bound']}", result['status']])
show_table(["Instance", "Literature distance", "Local conclusion", "Status"], rows)

| Instance | Literature distance | Local conclusion | Status |
| --- | --- | --- | --- |
| BP90 | 10 | D >= 7 | bounded |
| BP108 | 8 | D >= 7 | bounded |

## 6. What this establishes

We traced the quotient boundary, exposed two incorrect counting rules, checked
representative independence, and reproduced both paper matrices and ranks.
The distance searches give lower bounds only. Free actions are not required
for the BP constructor, although particular correspondences need stronger
hypotheses; see [04 — LP–BP and FB–BP](04-fb-and-correspondences.ipynb).

Implementation: [orbit constructor](../qldpc_generator/balanced.py),
[negative controls](../tests/test_products.py), and
[folded-matrix comparisons](../tests/test_paper_cases.py).